In [2]:
import pyspark

In [2]:

spark = SparkSession.builder \
    .appName("StockTweetForecasting") \
    .config("spark.pyspark.python", "/home/hduser/pyspark_env/bin/python") \
    .config("spark.pyspark.driver.python", "/home/hduser/pyspark_env/bin/python") \
    .config("spark.executorEnv.PYSPARK_PYTHON", "/home/hduser/pyspark_env/bin/python") \
    .getOrCreate()

26/05/26 12:52:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, avg
from pyspark.sql.functions import lower, regexp_replace, trim, length

spark = SparkSession.builder.appName("StockTweetForecasting").getOrCreate()

stock_tweets = spark.read.csv("hdfs:///stocks/raw_data/stocktweet.csv",header=True,inferSchema=True)

stock_tweets = stock_tweets.withColumn("date", to_date(col("date"), "dd/MM/yyyy"))

stock_tweets.printSchema()
stock_tweets.show(10, truncate=False)

26/05/26 13:05:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- ticker: string (nullable = true)
 |-- tweet: string (nullable = true)

+------------------------------------------------------------------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|id                                                                |date      |ticker|tweet                                                                                                                                      |
+------------------------------------------------------------------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|100001                                                            |2020-01-01|AMZN  |$AMZN Dow futures up by 100 points already 🥳                                        

In [2]:
import re
import html
import unicodedata
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def clean_text(text):

    if text is None:
        return ""

    
    text = str(text) # Convert to string
    text = html.unescape(text) # Fix HTML entities
    text = unicodedata.normalize("NFKD", text) # Normalize unicode
    text = text.encode("ascii", "ignore").decode("ascii") # Remove corrupted unicode/emojis
    text = text.lower()  # Lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text) # Remove URLs
    text = re.sub(r"\$[A-Za-z]+", " ", text) # Remove ticker symbols
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text) # Remove mentions
    text = re.sub(r"#", "", text) # Remove hashtags symbol only
    text = re.sub(r"\d+", " ", text) # Remove numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # Remove punctuation
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces
    return text




In [3]:
text_udf = udf(clean_text, StringType())

cleaned_df = stock_tweets.withColumn(
    "clean_text",
    text_udf(col("tweet"))
)

selected_tickers = ["ABNB", "AMZN", "NKE", "GOOGL", "NFLX"]

cleaned_df = cleaned_df.filter(
    col("ticker").isin(selected_tickers)
)

cleaned_df.select("ticker", "clean_text").show(30, truncate=False)

cleaned_df.select(
    "ticker",
    "clean_text"
).show(30, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticker|clean_text                                                                                                                                                                                              |
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|AMZN  |dow futures up by points already                                                                                                                                                                        |
|AMZN  |who ever shorted today will be deported from us if you are immigrants                                                                                   

In [1]:
%pip install vaderSentiment

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyspark.sql.functions import udf, col, when, count, avg
from pyspark.sql.types import FloatType

analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    if text is None:
        return 0.0
    return float(analyzer.polarity_scores(str(text))["compound"])

sentiment_udf = udf(get_sentiment_score, FloatType())

tweets_sentiment = cleaned_df.withColumn(
    "sentiment_score",
    sentiment_udf(col("clean_tweet"))
)

tweets_sentiment = tweets_sentiment.withColumn(
    "sentiment_label",
    when(col("sentiment_score") > 0.05, "positive")
    .when(col("sentiment_score") < -0.05, "negative")
    .otherwise("neutral")
)
